# Análise de viagens a serviço de servidores públicos federais (2025)

Dados abertos do [Portal da Transparência](https://portaldatransparencia.gov.br) (CGU): viagens a serviço e passagens emitidas.

**Perguntas que a análise responde**

1. Qual a proporção de viagens por cargo?
2. Qual a despesa média por viagem em cada órgão solicitante?
3. Quais cargos têm despesa média por viagem acima da média geral?
4. Como o número de viagens e os gastos variam ao longo do ano, e a duração da viagem explica o custo?

**Como rodar:** baixe os CSVs conforme `data/README.md`, ajuste `DATA_DIR` na célula de configuração e execute o notebook em ordem (*Executar tudo*).

## 1. Configuração

In [ ]:
import re
import unicodedata
from pathlib import Path
from textwrap import shorten

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ANO = 2025               # troque para reaproveitar a análise em outros anos
MIN_VIAGENS = 30         # mínimo de viagens para um órgão/cargo entrar nos rankings de média
LIMITE_ALTO = 200_000    # viagens acima desse valor (R$) são tratadas como casos extremos

# Pasta com os CSVs baixados do Portal da Transparência.
DATA_DIR = Path("../data")
IMG_DIR = Path("../images")
IMG_DIR.mkdir(exist_ok=True)

# No Google Colab, descomente as linhas abaixo e aponte para a sua pasta no Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = Path("/content/drive/MyDrive/Projeto Portal da Transparência Viagens")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COR = "#1f77b4"

### Funções auxiliares

`para_snake_case` padroniza os nomes das colunas (os CSVs têm acentos, espaços e símbolos) e `brl` formata valores no padrão brasileiro (R$ 1.234,56).

In [ ]:
def para_snake_case(texto):
    """'Período - Data de início' -> 'periodo_data_de_inicio'."""
    texto = unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("ASCII")
    texto = re.sub(r"[^a-zA-Z0-9\s]", "", texto)
    return re.sub(r"\s+", "_", texto.strip()).lower()


def brl(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(valor):
    return f"{valor:.2f}%".replace(".", ",")


def inteiro(valor):
    return f"{valor:,.0f}".replace(",", ".")


def decimal(valor, casas=2):
    return f"{valor:.{casas}f}".replace(".", ",")


def formatar(df, moeda=(), porcentagem=(), dias=(), inteiros=()):
    """Devolve uma cópia do DataFrame com colunas formatadas (apenas para exibição)."""
    out = df.copy()
    for c in moeda:
        out[c] = out[c].map(brl)
    for c in porcentagem:
        out[c] = out[c].map(pct)
    for c in dias:
        out[c] = out[c].map(lambda x: f"{decimal(x, 1)} dias")
    for c in inteiros:
        out[c] = out[c].map(inteiro)
    return out


def rotulo(texto, n=45):
    return shorten(str(texto), width=n, placeholder="…")

## 2. Carregamento dos dados

Dois arquivos do Portal: **viagens** (uma linha por viagem) e **passagens** (trechos emitidos de cada viagem). Só as colunas necessárias são lidas, o que economiza memória. As colunas `nome` e `cpf_viajante` **não são carregadas**: a análise é agregada e não precisa identificar pessoas.

In [ ]:
COLUNAS_VIAGEM = [
    "identificador_do_processo_de_viagem", "nome_orgao_solicitante", "cargo", "destinos",
    "periodo_data_de_inicio", "periodo_data_de_fim",
    "valor_diarias", "valor_passagens", "valor_devolucao", "valor_outros_gastos",
]
COLUNAS_PASSAGEM = [
    "identificador_do_processo_de_viagem", "meio_de_transporte", "pais_destino_ida",
    "cidade_origem_ida", "cidade_destino_ida", "valor_da_passagem", "taxa_de_servico",
]


def ler_csv(caminho, colunas):
    df = pd.read_csv(
        caminho, sep=";", encoding="cp1252", decimal=",", low_memory=False,
        usecols=lambda c: para_snake_case(c) in colunas,
    )
    df.columns = [para_snake_case(c) for c in df.columns]
    return df


df = ler_csv(DATA_DIR / f"{ANO}_Viagem.csv", COLUNAS_VIAGEM)
df_passagem = ler_csv(DATA_DIR / f"{ANO}_Passagem.csv", COLUNAS_PASSAGEM)

print(f"Viagens: {inteiro(len(df))} linhas | Passagens: {inteiro(len(df_passagem))} linhas")
df.head(3)

## 3. Tratamento e novas variáveis

- Valores monetários viram `float` (se algum vier como texto, a vírgula decimal é convertida).
- Datas viram `datetime`; datas inválidas ficam como `NaT`.
- Cargo ausente é explicitado como `Não informado`.
- `despesas_totais` = diárias + passagens + outros gastos.
- `dias_de_viagem` conta o dia de partida e o de retorno (uma viagem de 1 a 3 do mês tem **3** dias).

In [ ]:
COLUNAS_VALOR = ["valor_diarias", "valor_passagens", "valor_devolucao", "valor_outros_gastos"]

for col in COLUNAS_VALOR:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="coerce")

for col in ["periodo_data_de_inicio", "periodo_data_de_fim"]:
    df[col] = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")

df["cargo"] = df["cargo"].fillna("Não informado")

df["despesas_totais"] = df[["valor_diarias", "valor_passagens", "valor_outros_gastos"]].sum(axis=1)

dias = (df["periodo_data_de_fim"] - df["periodo_data_de_inicio"]).dt.days + 1
df["dias_de_viagem"] = dias.where(dias >= 1)  # datas ausentes ou fim antes do início viram NaN

for col in ["valor_da_passagem", "taxa_de_servico"]:
    if not pd.api.types.is_numeric_dtype(df_passagem[col]):
        df_passagem[col] = pd.to_numeric(df_passagem[col].str.replace(",", ".", regex=False), errors="coerce")

### Verificações de qualidade

In [ ]:
qualidade = pd.Series({
    "Viagens": len(df),
    "Sem data de início ou fim válida": df["dias_de_viagem"].isna().sum(),
    "Com despesa total igual a zero": (df["despesas_totais"] == 0).sum(),
    "Cargo 'Não informado'": (df["cargo"] == "Não informado").sum(),
})
display(formatar(qualidade.rename("quantidade").to_frame(), inteiros=["quantidade"]))

share_devolucao = df["valor_devolucao"].sum() / df["despesas_totais"].sum() * 100
print(f"Devoluções somam {pct(share_devolucao)} das despesas totais (não são descontadas nos cálculos).")
resumo = df[["valor_diarias", "valor_passagens", "valor_outros_gastos", "despesas_totais"]].describe().drop(index="count")
display(resumo.apply(lambda c: c.map(brl)))

## 4. Análise

### 4.1 Proporção de viagens por cargo

In [ ]:
prop_cargo = (
    df["cargo"].value_counts(normalize=True).mul(100)
    .rename("proporcao_pct").reset_index()
)
display(formatar(prop_cargo.head(10), porcentagem=["proporcao_pct"]))

top = prop_cargo.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["cargo"].map(lambda c: rotulo(c, 40)), top["proporcao_pct"], color=COR)
ax.set_title("Proporção de viagens por cargo (10 maiores)", fontweight="bold")
ax.set_xlabel("% das viagens")
for y, v in enumerate(top["proporcao_pct"]):
    ax.text(v + 0.3, y, pct(v), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(IMG_DIR / "01_proporcao_cargos.png", bbox_inches="tight")
plt.show()

> **Atenção:** "Não informado" (campo vazio) e "Informações protegidas por sigilo" são categorias que **não** descrevem um cargo. Elas ocupam as primeiras posições e limitam o que a análise por cargo consegue dizer.

### 4.2 Despesa média por órgão solicitante

Só entram órgãos com pelo menos `MIN_VIAGENS` viagens, para que médias de órgãos com pouquíssimas viagens não distorçam o ranking. A mediana é exibida porque a distribuição é muito assimétrica (muitas viagens não têm passagem e poucas têm valores altíssimos).

In [ ]:
por_orgao = (
    df.groupby("nome_orgao_solicitante")
    .agg(
        n_viagens=("despesas_totais", "size"),
        despesa_media=("despesas_totais", "mean"),
        despesa_mediana=("despesas_totais", "median"),
        despesa_total=("despesas_totais", "sum"),
        dias_medio=("dias_de_viagem", "mean"),
    )
    .reset_index()
)
ranking_orgao = por_orgao[por_orgao["n_viagens"] >= MIN_VIAGENS].sort_values("despesa_media", ascending=False)
print(f"{len(ranking_orgao)} de {len(por_orgao)} órgãos têm {MIN_VIAGENS}+ viagens.")

display(formatar(
    ranking_orgao.head(10).assign(nome_orgao_solicitante=lambda d: d["nome_orgao_solicitante"].map(lambda x: rotulo(x, 55))),
    moeda=["despesa_media", "despesa_mediana", "despesa_total"], dias=["dias_medio"], inteiros=["n_viagens"],
))

top = ranking_orgao.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["nome_orgao_solicitante"].map(lambda c: rotulo(c, 40)), top["despesa_media"], color=COR)
ax.set_title(f"Despesa média por viagem: 10 maiores órgãos ({MIN_VIAGENS}+ viagens)", fontweight="bold")
ax.set_xlabel("R$ por viagem")
plt.tight_layout()
plt.savefig(IMG_DIR / "02_despesa_media_orgao.png", bbox_inches="tight")
plt.show()

### 4.3 Cargos com despesa média acima da média geral

In [ ]:
media_geral = df["despesas_totais"].mean()

por_cargo = (
    df.groupby("cargo")
    .agg(
        n_viagens=("despesas_totais", "size"),
        despesa_media=("despesas_totais", "mean"),
        despesa_mediana=("despesas_totais", "median"),
        despesa_total=("despesas_totais", "sum"),
    )
    .reset_index()
)
elegiveis = por_cargo[por_cargo["n_viagens"] >= MIN_VIAGENS]
acima_media = elegiveis[elegiveis["despesa_media"] > media_geral].sort_values("despesa_media", ascending=False)

print(f"Média geral por viagem: {brl(media_geral)}")
print(f"{len(acima_media)} de {len(elegiveis)} cargos ({MIN_VIAGENS}+ viagens) estão acima da média geral.")
display(formatar(
    acima_media.head(10).assign(cargo=lambda d: d["cargo"].map(lambda x: rotulo(x, 55))),
    moeda=["despesa_media", "despesa_mediana", "despesa_total"], inteiros=["n_viagens"],
))

top = acima_media.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["cargo"].map(lambda c: rotulo(c, 40)), top["despesa_media"], color=COR)
ax.axvline(media_geral, color="crimson", linestyle="--", label=f"Média geral: {brl(media_geral)}")
ax.set_title("Cargos com maior despesa média por viagem", fontweight="bold")
ax.set_xlabel("R$ por viagem")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(IMG_DIR / "03_cargos_acima_da_media.png", bbox_inches="tight")
plt.show()

### 4.4 Evolução ao longo do ano e relação entre duração e custo

In [ ]:
no_ano = df[df["periodo_data_de_inicio"].dt.year == ANO]
mensal = (
    no_ano.groupby(no_ano["periodo_data_de_inicio"].dt.month)
    .agg(n_viagens=("despesas_totais", "size"), despesa_total=("despesas_totais", "sum"))
    .reindex(range(1, 13), fill_value=0)
)
meses = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(meses, mensal["n_viagens"], marker="o", color=COR, linewidth=2)
axes[0].set_title("Viagens iniciadas por mês", fontweight="bold")
axes[0].set_ylabel("Nº de viagens")
axes[1].bar(meses, mensal["despesa_total"] / 1e6, color=COR)
axes[1].set_title("Despesa total por mês", fontweight="bold")
axes[1].set_ylabel("R$ milhões")
for ax in axes:
    ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(IMG_DIR / "04_evolucao_mensal.png", bbox_inches="tight")
plt.show()

In [ ]:
validas = df[df["dias_de_viagem"].notna() & (df["despesas_totais"] > 0)]
corr = validas["dias_de_viagem"].corr(validas["despesas_totais"], method="spearman")
amostra = validas.sample(min(20_000, len(validas)), random_state=42)  # amostra: 800 mil pontos não se leem num scatter

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(amostra["dias_de_viagem"], amostra["despesas_totais"], alpha=0.2, s=10, color=COR)
ax.set_yscale("log")
ax.set_title(f"Duração × despesa total (amostra de {inteiro(len(amostra))}; Spearman = {decimal(corr)})", fontweight="bold")
ax.set_xlabel("Dias de viagem")
ax.set_ylabel("Despesa total (R$, escala log)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(IMG_DIR / "05_duracao_vs_despesa.png", bbox_inches="tight")
plt.show()

### 4.5 Casos extremos

Viagens acima de `LIMITE_ALTO`. Os nomes dos viajantes não são carregados; a tabela mostra apenas órgão, cargo e destino. Para a mais cara, os dados de passagens mostram de onde vem o custo.

In [ ]:
extremos = df[df["despesas_totais"] > LIMITE_ALTO].sort_values("despesas_totais", ascending=False)
print(f"{inteiro(len(extremos))} viagem(ns) acima de {brl(LIMITE_ALTO)} ({pct(len(extremos) / len(df) * 100)} do total).")

colunas = ["nome_orgao_solicitante", "cargo", "destinos", "dias_de_viagem",
           "valor_diarias", "valor_passagens", "valor_outros_gastos", "despesas_totais"]
display(formatar(
    extremos[colunas].head(10).assign(
        nome_orgao_solicitante=lambda d: d["nome_orgao_solicitante"].map(lambda x: rotulo(x, 35)),
        destinos=lambda d: d["destinos"].map(lambda x: rotulo(x, 35)),
    ),
    moeda=["valor_diarias", "valor_passagens", "valor_outros_gastos", "despesas_totais"],
    dias=["dias_de_viagem"],
))

id_maior = df.loc[df["despesas_totais"].idxmax(), "identificador_do_processo_de_viagem"]
passagens_maior = df_passagem[df_passagem["identificador_do_processo_de_viagem"] == id_maior]
print(f"\nPassagens da viagem mais cara (id {id_maior}): {len(passagens_maior)} trecho(s)")
display(formatar(
    passagens_maior.drop(columns="identificador_do_processo_de_viagem"),
    moeda=["valor_da_passagem", "taxa_de_servico"],
))

## 5. Principais achados

O texto abaixo é gerado a partir dos resultados das células anteriores, então se mantém correto ao trocar o `ANO`.

In [ ]:
p = prop_cargo.set_index("cargo")["proporcao_pct"]
nao_inf = p.get("Não informado", 0.0)
sigilo = p.get("Informações protegidas por sigilo", 0.0)
orgao_top = ranking_orgao.iloc[0]
cargo_top = acima_media.iloc[0] if len(acima_media) else None
mes_pico = meses[int(mensal["n_viagens"].idxmax()) - 1]
mes_gasto = meses[int(mensal["despesa_total"].idxmax()) - 1]
forca = "fraca" if abs(corr) < 0.3 else "moderada" if abs(corr) < 0.6 else "forte"

linhas = [
    f"- **Volume:** {inteiro(len(df))} viagens em {ANO}, com despesa total de {brl(df['despesas_totais'].sum())} (média de {brl(media_geral)} por viagem; mediana de {brl(df['despesas_totais'].median())}).",
    f"- **Cargos:** {pct(nao_inf)} das viagens não têm cargo informado e {pct(sigilo)} estão sob sigilo; juntas, {pct(nao_inf + sigilo)} das viagens.",
    f"- **Órgãos:** entre os órgãos com {MIN_VIAGENS}+ viagens, a maior despesa média é de **{orgao_top['nome_orgao_solicitante']}** ({brl(orgao_top['despesa_media'])} por viagem).",
]
if cargo_top is not None:
    linhas.append(f"- **Cargos acima da média:** {len(acima_media)} de {len(elegiveis)} cargos superam a média geral; o maior é **{cargo_top['cargo']}** ({brl(cargo_top['despesa_media'])} por viagem).")
linhas += [
    f"- **Sazonalidade:** o maior número de viagens começa em **{mes_pico}** e o maior gasto ocorre em **{mes_gasto}**.",
    f"- **Duração × custo:** correlação {forca} (Spearman = {decimal(corr)}); viagens mais longas tendem a custar mais, mas a duração explica só parte do gasto.",
    f"- **Casos extremos:** {inteiro(len(extremos))} viagem(ns) superam {brl(LIMITE_ALTO)}.",
]
display(Markdown("\n".join(linhas)))

## 6. Limitações

- A despesa total é **bruta**: não desconta devoluções (ver verificação de qualidade).
- Cargos "Não informado" e "sigilo" concentram boa parte das viagens, então a análise por cargo é parcial.
- A média é sensível a valores extremos; por isso as tabelas trazem também a mediana.
- Cada linha é um registro de viagem do Portal; o notebook não distingue viagens nacionais de internacionais.
- O Portal atualiza os arquivos periodicamente; resultados podem mudar conforme a data de download.